# RDS → S3 daily pipeline

Notebook này deploy và chạy thử pipeline daily sau DMS bootstrap. Daily job dùng window `(watermark, cutoff]`, Glue JDBC và Hive partitions `year/month/day`.

## 1. Chuẩn bị

Copy `.env.example` thành `.env`. `DMS_PREFIX` phải trỏ tới task initial. `setup()` tự đọc cutoff thật từ DMS và kiểm tra curated target; schedule nên giữ `ENABLE_SCHEDULE=false` tới khi test thành công.

In [1]:
%pip install boto3 python-dotenv -q

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [10]:
import importlib
import sys
from pathlib import Path

notebook_dir = Path.cwd()
if not (notebook_dir / 'deploy.py').is_file():
    notebook_dir = Path.cwd() / 'rds_daily_pipeline'
sys.path.insert(0, str(notebook_dir))
sys.modules.pop('deploy', None)  # Tránh import nhầm deploy.py của pipeline khác.
import deploy

importlib.reload(deploy)
cfg = deploy.config()
print(f'Region     : {cfg.region}')
print(f'Source     : {cfg.schema}.{cfg.table}')
print(f'Date column: {cfg.date_column}')
print(f'DMS prefix : {cfg.dms_prefix}')
print(f'Raw        : s3://{cfg.bucket}/{cfg.raw_prefix}/year=YYYY/month=MM/day=DD/')
print(f'Curated    : s3://{cfg.bucket}/{cfg.curated_prefix}/year=YYYY/month=MM/day=DD/')
print(f'Schedule on: {cfg.enable_schedule}')
print(f'Primary key: {cfg.primary_key}')
print(f'Child      : {cfg.child_tables}')
print(f'Purge      : enabled={cfg.enable_purge} dry_run={cfg.purge_dry_run}')

Region     : ap-southeast-1
Source     : public.orders
Date column: created_at_utc
DMS prefix : orders-initial
Raw        : s3://my-data-lake-lklklklkklkiet/raw/rds-daily/orders/year=YYYY/month=MM/day=DD/
Curated    : s3://my-data-lake-lklklklkklkiet/curated/rds/orders/year=YYYY/month=MM/day=DD/
Schedule on: False


## 2. Deploy/update

Tạo Glue JDBC connection trong private subnet, Glue job, crawler, control table, Step Functions và EventBridge. RDS security group phải cho phép Glue security group vào port PostgreSQL.

In [11]:
deploy.setup()  # Bỏ dấu # sau khi kiểm tra config.

Watermark already exists: 2026-05-18T00:00:00Z
Glue network: subnet=subnet-039095fa2ef6079a9, security_groups=['sg-0fe28ad984d087a6a']
Daily RDS pipeline ready: arn:aws:states:ap-southeast-1:637423316258:stateMachine:orders-rds-daily-workflow
Schedule: DISABLED (cron(0 2 * * ? *))


## 3. Chạy thử và theo dõi

Watermark chỉ được commit sau khi Glue job và crawler thành công. Nếu execution lỗi, sửa nguyên nhân rồi chạy lại cùng window.

In [12]:
execution_arn = deploy.run_once()

Execution started: arn:aws:states:ap-southeast-1:637423316258:execution:orders-rds-daily-workflow:fbd9a996-f500-46fe-99b7-cb329799804c


In [13]:
deploy.execution_status(execution_arn)

C:\Users\sirtu\AppData\Roaming\Python\Python39\site-packages\boto3\compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


Status : SUCCEEDED
Started: 2026-08-17 00:06:38.068000+07:00
Stopped: 2026-08-17 00:06:39.006000+07:00


{'executionArn': 'arn:aws:states:ap-southeast-1:637423316258:execution:orders-rds-daily-workflow:fbd9a996-f500-46fe-99b7-cb329799804c',
 'stateMachineArn': 'arn:aws:states:ap-southeast-1:637423316258:stateMachine:orders-rds-daily-workflow',
 'name': 'fbd9a996-f500-46fe-99b7-cb329799804c',
 'status': 'SUCCEEDED',
 'startDate': datetime.datetime(2026, 8, 17, 0, 6, 38, 68000, tzinfo=tzlocal()),
 'stopDate': datetime.datetime(2026, 8, 17, 0, 6, 39, 6000, tzinfo=tzlocal()),
 'input': '{}',
 'inputDetails': {'included': True},
 'output': '{"skip":true,"reason":"No eligible archive window"}',
 'outputDetails': {'included': True},
 'redriveCount': 0,
 'redriveStatus': 'NOT_REDRIVABLE',
 'redriveStatusReason': 'Execution is SUCCEEDED and cannot be redriven',
 'ResponseMetadata': {'RequestId': '40df97bf-0cbf-422b-9590-8b4b234b6fcc',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amzn-requestid': '40df97bf-0cbf-422b-9590-8b4b234b6fcc',
   'date': 'Sun, 16 Aug 2026 17:07:42 GMT',
   'content-type':

## 4. Bật purge (xóa dữ liệu khỏi RDS)

Archive không tự làm RDS nhỏ lại. State `PurgeSource` chạy Glue job `<PIPELINE_NAME>-purge-source`: verify từng key của window trong curated Parquet (anti-join + checksum + child rows), chỉ khi PASSED mới xóa child rồi parent theo batch.

Đi ba bước, mỗi bước chạy lại `deploy.setup()`:

1. `ENABLE_PURGE=false` — chỉ archive, xác nhận curated data đúng.
2. `ENABLE_PURGE=true` + `PURGE_DRY_RUN=true` — log Glue in `Verify: PASSED` và số row sẽ xóa, RDS chưa mất gì.
3. `PURGE_DRY_RUN=false` — xóa thật. Không quay lại được, hãy chắc bước 2 đã PASSED.

## 5. Bật daily schedule

Sau lần test `SUCCEEDED`, đổi `ENABLE_SCHEDULE=true` trong `.env`, rồi chạy lại `deploy.setup()`. EventBridge cron dùng UTC.

## 6. Destroy pipeline

Xóa orchestration/compute do pipeline tạo. **Giữ nguyên RDS, S3 raw/curated, Glue database và catalog tables.**

In [ ]:
# deploy.destroy()  # Bỏ dấu # chỉ khi thật sự muốn teardown.